In [ ]:
!pip install -q faiss-cpu sentence-transformers streamlit numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 107.9 MB/s eta 0:00:00


In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [ ]:
text = """
Pakistan University Policy:
Attendance minimum 75% required.
Grading: 85% and above is A grade.
Semester starts in September.
Refund allowed only in first week.
"""

In [ ]:
def chunk_text(text, size=30):
    words = text.split()
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size)]

chunks = chunk_text(text)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vectors = model.encode(chunks)
vectors = np.array(vectors).astype("float32")

In [ ]:
index = faiss.IndexFlatL2(vectors.shape[1])
index.add(vectors)

In [ ]:
def retrieve(query, k=2):
    q_vec = model.encode([query]).astype("float32")
    _, idx = index.search(q_vec, k)
    return [chunks[i] for i in idx[0]]

In [ ]:
chat_history = []

In [ ]:
def chat(query):
    context = retrieve(query)

    answer = " ".join(context)

    chat_history.append((query, answer))

    return answer

In [ ]:
print(chat("What is attendance requirement?"))
print(chat("What is grading policy?"))

Pakistan University Policy: Attendance minimum 75% required. Grading: 85% and above is A grade. Semester starts in September. Refund allowed only in first week. Pakistan University Policy: Attendance minimum 75% required. Grading: 85% and above is A grade. Semester starts in September. Refund allowed only in first week.
Pakistan University Policy: Attendance minimum 75% required. Grading: 85% and above is A grade. Semester starts in September. Refund allowed only in first week. Pakistan University Policy: Attendance minimum 75% required. Grading: 85% and above is A grade. Semester starts in September. Refund allowed only in first week.


In [ ]:
%%writefile app.py

import streamlit as st

st.title("My RAG Chatbot")

st.write("Hello World")

Writing app.py


In [ ]:
!streamlit run app.py > /content/streamlit.log 2>&1 &